In [25]:
import os
from pathlib import Path
from unicodedata import category
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import statistics
import pandas as pd
import duckdb
import numpy as np
import functools
import pyarrow
import itertools
import json



In [26]:
os.getcwd()

'/Users/eric/Documents/SchoolCourses/PaGamO/analyze/IRT'

In [27]:
df_ans_log = pd.read_parquet("../../data/IRT/40460_gc_answer_log 1.parquet")
df_question_struct = pd.read_parquet("../../data/IRT/40460_question_structure.parquet")
df_user_info = pd.read_parquet("../../data/IRT/40460_user_info.parquet")

In [28]:
# merge all datas by question id and user id
df = df_ans_log.copy()
df = pd.merge(df,df_question_struct[['question_id','answer','book_volume_grade','section_name','book_volume_year','difficulty_level']],how = 'left' , on= 'question_id')
df = pd.merge(df,df_user_info[['user_id','user_grade']],how = 'left', on= 'user_id')
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df = df.sort_values(by='user_id')
df_cp = df.copy()


In [29]:
df_cp.section_name.value_counts()

section_name
1 10000以內的數                    7915507
9 分數                           3413485
8 分數                           3192847
2 四位數的加減                       2958755
4 乘法                           2878249
                                ...   
第4章 正方體和長方體4-1 正方體和長方體的構成要素          7
03垂直平分線與角平分線                         5
03全等三角形的應用                           5
01二次函數及其圖形                           5
02三角形的全等                             4
Name: count, Length: 1096, dtype: int64

In [30]:
df_cp.columns

Index(['is_correct', 'created_at', 'id', 'question_id', 'gamecharacter_id',
       'user_id', 'answer', 'book_volume_grade', 'section_name',
       'book_volume_year', 'difficulty_level', 'user_grade'],
      dtype='object')

In [31]:
df.value_counts('book_volume_grade')

book_volume_grade
3    46371678
4    19869915
5    19244059
6    10757281
7     1446361
8      248715
9       47378
Name: count, dtype: int64

In [32]:
df = df_cp.copy()

In [45]:
GRADE = 5

In [46]:
# filter the certain section
top_sections = df.query(f'book_volume_grade == {GRADE}').value_counts('section_name').index.to_list()[:]
top_sections = top_sections if type(top_sections) is list else [top_sections]
print(len(top_sections))

# save topsection to list
with open('../../data/processed/top_section_lis.json','w') as f:
    json.dump(top_sections,f)

67


In [47]:
# df = df.query(f"book_volume_grade == {GRADE} and user_grade == {GRADE} and section_name == {top_sections}")
df = df.query(f"book_volume_grade == {GRADE} and user_grade == {GRADE}")

In [48]:
dic_of_index = df.value_counts("user_id").copy()
dic_of_index = dic_of_index.sort_index().cumsum().to_list()
dic_of_index[-1]

749699

In [49]:
cutpts = [i for i in itertools.islice(dic_of_index,None,None,5000)]

if cutpts[-1] != dic_of_index[-1]:
    cutpts.append(dic_of_index[-1])

dfs = [df[cutpts[idx]:cutpts[idx+1]].copy() for idx in range(len(cutpts) - 1)]
len(dfs)

4

In [50]:
#TEMP_DIR = "/Volumes/Crucial_X6/duckdb"
con = duckdb.connect()
con.execute("SET threads=2;")
#con.execute(f"PRAGMA temp_directory='{TEMP_DIR}';")
con.execute("SET memory_limit='16GB';")


In [51]:
df_sessions = []
for i,idf in enumerate(dfs):
    print(f"processing {i+1} / {len(dfs)}")
    con.register(f"df",idf)
    query = """
    WITH base AS (
        SELECT
            user_id,
            gamecharacter_id,
            is_correct,
            difficulty_level,
            question_id,
            section_name,
            created_at,
            answer,
            book_volume_grade,
            -- 針對 (user_id, gamecharacter_id) 做 lag
            LAG(created_at) OVER (
                PARTITION BY user_id,section_name
                ORDER BY created_at
            ) AS prev_time
        FROM df
    ),
    step1 AS (
        SELECT
            *,
            -- 秒數差：created_at - prev_time
            date_diff('second', prev_time, created_at) AS time_diff,
            CASE
                WHEN prev_time IS NULL
                  OR date_diff('second', prev_time, created_at) > 30 * 60
                THEN 1 ELSE 0
            END AS new_session_flag
        FROM base
    ),
    step2 AS (
        SELECT
            *,
            -- 依 user_id 做 new_session_flag 的 cumulative sum
            SUM(new_session_flag) OVER (
                PARTITION BY user_id, section_name
                ORDER BY created_at
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS session_id
        FROM step1
    ),
    session_agg AS (
        SELECT
            user_id,
            session_id,
            section_name,
            -- 這裡 user_id / session_id 本來就 group key，可直接留著；示意上還是給你一個 mode 寫法
            MODE(gamecharacter_id) AS gamecharacter_id_mode,

            -- 以下用 list 聚合，並依 created_at 排序，等同你原本的 list（而且保持時間順序）
            LIST(is_correct ORDER BY created_at)        AS is_correct,
            LIST(difficulty_level ORDER BY created_at)  AS difficulty_level,
            LIST(question_id ORDER BY created_at)       AS question_id,
            LIST(created_at ORDER BY created_at)        AS created_at,
            LIST(answer ORDER BY created_at)            AS answer,
            LIST(book_volume_grade ORDER BY created_at) AS book_volume_grade,
            LIST(time_diff ORDER BY created_at)         AS time_diff,

            MIN(created_at) AS start_time,
            MAX(created_at) AS end_time,
            -- session 長度（秒）
            date_diff('second', MIN(created_at), MAX(created_at)) AS session_length,
            -- answer 的長度
            COUNT(answer) AS answer_length,
        FROM step2
        GROUP BY user_id, session_id, section_name
    )
    SELECT * FROM session_agg
    """
    df_temp = con.execute(query).df().copy()
    df_sessions.append(df_temp)
    con.unregister("df")



processing 1 / 4
processing 2 / 4
processing 3 / 4
processing 4 / 4


In [52]:
df_session = pd.concat(df_sessions)

In [53]:
df_session = df_session.reset_index()

In [54]:
df_session.to_parquet('../../data/processed/irt_session.parquet',engine='pyarrow')

In [55]:
# def construct_session_id(df:pd.DataFrame):
#     df.loc[:,'created_at'] = pd.to_datetime(df['created_at'], errors = 'coerce')
#     df.sort_values(by = ['created_at'],inplace=True)
#     df.loc[:,'prev_time'] = df.groupby(['user_id','gamecharacter_id'])['created_at'].shift(1)
#     df.loc[:,'time_diff'] = (df['created_at'] - df['prev_time']).dt.total_seconds()
#     df.loc[:,'new_session_flag'] = ((df['time_diff'].copy() > 30*60 ) | df['time_diff'].copy().isna()).astype(int)
#     df.loc[:,'session_id'] = df.groupby("user_id")["new_session_flag"].cumsum()
#     return df
# def agg_sessions(df:pd.DataFrame) -> pd.DataFrame:
#     session_df = df.groupby(['user_id','session_id']).agg({
#         'user_id':pd.Series.mode,
#         'gamecharacter_id':pd.Series.mode,
#         'session_id': pd.Series.mode,
#         'is_correct':list,
#         'difficulty_level':list,
#         'question_id':list,
#         'section_name':list,
#         'created_at':list,
#         'answer':list,
#         'book_volume_grade':list,
#         'time_diff':list,
#     })
#     session_df.loc[:,'start_time'] = session_df['created_at'].apply(lambda x: min(x))
#     session_df.loc[:,'end_time'] = session_df['created_at'].apply(lambda x: max(x))
#     session_df.loc[:,'session_length'] = (session_df['end_time'] - session_df['start_time']).dt.total_seconds()
#     session_df.loc[:,'answer_length'] = session_df['answer'].apply(lambda x: len(x))
#
#     # try:
#     #     session_df.loc[:,'section_name_mode'] = session_df['section_name'].apply(lambda x: pd.Series(x).mode())
#     # except ValueError:
#     #     print("section_name list have more than one mode")
#     #     session_df.loc[:, 'section_name_mode'] = session_df['section_name'].apply(lambda x: pd.Series(x).mode()[0])
#
#     """
#     temp = []
#     for idx,row in session_df.iterrows():
#         try:
#             print(row['experiment_index'])
#             temp.append(pd.Series(row['experiment_index']).mode())
#         except:
#             print(row['experiment_index'])
#             temp.append(-99)
#
#     session_df.loc[:,'experiment_mode'] = temp
#     """
#     return session_df


In [56]:
# merge polars is no efficient
# import polars as pl
# df_ans_log = pl.read_parquet("../../data/IRT/40460_gc_answer_log 1.parquet")
# df_question_struct = pl.read_parquet("../../data/IRT/40460_question_structure.parquet")
# df_user_info = pl.read_parquet("../../data/IRT/40460_user_info.parquet")
#
#
# # 假設這三個本來就是 pl.DataFrame
# df = df_ans_log.clone()  # 類似 pandas 的 copy()
#
# df = (
#     df
#     .join(df_question_struct, on="question_id", how="left")
#     .join(df_user_info,      on="user_id",     how="left")
# )
# df.write_parquet("../../data/processed/temp/polars.parquets")
#
# df = pd.read_parquet("../../data/processed/temp/polars.parquets")
# df.info()